# Understanding Imputation in Shapley Value Explanations: Gaussian and Gaussian Copula Data Imputers 

## Introduction

This notebook demonstrates how to use Gaussian and Gaussian Copula imputation methods to handle missing feature values in the context of computing Shapley values for model interpretability. Since most models cannot handle incomplete data, these imputation techniques are used to fill in missing values. This enables reliable model-agnostic explanations, even when only partial input information is available.

## Gaussian Imputation (Theory)

We have a Shapley Value for each feature i, which is defined as

$$
\varphi_j = \sum_{S \subseteq M \setminus \{j\}} \frac{|S|! \, (M - |S| - 1)!}{M!} (v(S \cup \{j\}) - v(S))
$$

where $M$ is the set of all features and $S$ a subset of features not including feature $j$ . $v(S)$ is the model's prediction using only the features in $S$. Equivalent to this $v(S \cup \{j\})$ is the prediction when feature $j$ is added and $v(S \cup \{j\}) - v(S)$ is the marginal contribution of feature $j$ when added to subset $S$. 

With the Gaussian Imputation our goal is to approximate the value function $v(S)$ by using the statistical properties of the feature distribution. As proposed by [\[Aas21\]](../citations.rst) the feature vector $x$ follows a multivaiate Gaussian Distribution $\mathcal{N}_M(\boldsymbol{\mu}, \sum)$, estimated sing sample mean nd covariance from training data.


### The Key Properties and Implementation 

#### Gaussian Distribution

When the features are parted into observed set $S$ and unobserved set $\overline{S} = M \setminus S$, the conditional distribution of $x_{\overline{S}}$ given $x_S= x_S ^*$ is:

$$
p(x_{\overline{S}}| x_S=x_s^*) = \mathcal{N}_{|\overline{S}|}(\boldsymbol{\mu}_{\overline{S}|S}, \sum_{\overline{S}|S})
$$

where:

$$
\boldsymbol{\mu}_{\overline{S}|S} = \boldsymbol{\mu}_{\overline{S}} + \sum_{\overline{S}|S} \sum_{S|S}^{-1} (x_S^* - \boldsymbol{\mu}_S) 
$$

$$
\sum_{\overline{S}|S} = \sum_{\overline{S}|\overline{S}} - \sum_{\overline{S}|S} \sum_{S|S}^{-1} \sum_{S|\overline{S}}
$$

$\boldsymbol{\mu}_{\overline{S}|S}$ is the conditional mean, representing the most likely value for $\overline{S}$ given $S$. It adjusts the average of the missing features based on their correlation with the observed ones and how the observed values deviate from their mean. [\[Jul25\]](../citations.rst)

#### Monte Carlo Sampling 

We approximate $v(S)$ by drawing $K$ samples from the conditional distribution:

$$
v(S) = \frac{1}{K}  \sum_{k=1}^{K} f(x_{\overline{S}}^{(k)}, x_S^*) = \hat{v}(S)
$$

Each sample $x_{\overline{S}}^{(k)}$ represents a statistically plausible completion of the missing features given the observed $x_S^*$ and feature correlations. The value function $v(S)$ is approximated by averaging model predictions across these completions. We call $x^{(k)}$ a completion because it "completes" the partially observed instance $x_S^*$ into a full input for the model $f$.

#### Gaussian Copula Distribution
The Gaussian Copula method is used when the data does not follow a multivariate Gaussian distribution. It allows us to model the dependencies among features without requiring the marginal distributions to be Gaussian.

1. Transform each feature to gaussian space using the empirical cumulative distribution function (CDF):
    - $v_j = \Phi^{-1}(\hat{F}_j(x_j))$
2. Perform Gaussian imputation in this transformed space
3. Transform back to original space using the empirical quantile function
    - $\hat{x}_j = \hat{F}_j^{-1}(\Phi(v_j))$

## From Theory to Code

Having established the theoretical foundation, we now turn to its implementation, showing how to compute the conditional mean and covariance in practice.
But first we need to import modules, load the data and train the model. We are going to use the **Random Forest Regressor Model** and the **California housing dataset** for this example.

_This is adapted from the Conditional Imputer Notebook_

In [9]:
# Import Modules
import shapiq
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

print(f"shapiq version: {shapiq.__version__}")

shapiq version: 1.3.0


In [10]:
# Load Data
from shapiq.datasets import load_california_housing

X, y = load_california_housing()

# Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X.values,
    y.values,
    test_size=0.25,
    random_state=42,
)
n_features = X_train.shape[1]

In [11]:
# Train Model (Random Forest Regressor)
model = RandomForestRegressor(
    n_estimators=500,
    max_depth=n_features,
    max_features=2 / 3,
    max_samples=2 / 3,
    random_state=42,
)
model.fit(X_train, y_train)

,n_estimators,500
,criterion,'squared_error'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.6666666666666666
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Gaussian Imputation for Model Explanations
The first imputer we are going to look at is the `GaussianImputer` from `shapiq_student`. This imputer assumes a multivariate Gaussian distribution. 

It has following Key Attributes:
- `model`: The model to explain as a callable function expecting data points as input and returning the model's predictions.

- `data`: The background data to use for the explainer as a `np.ndarray` of shape `(n_samples, n_features)`.

- `x`: The explanation point as a `np.ndarray` of shape `(1, n_features)` or `(n_features,)`. Defaults to `None`.

- `n_mc_samples`: Number of Monte Carlo samples for imputation. Defaults to 1000.

- `random_state`: The random state to use for sampling. Defaults to ``None``.

### Initialize Gaussian Imputer:

In [ ]:
from shapiq_student import GaussianImputer

# 1. Create custom Gaussian imputer
gaussian_imputer = GaussianImputer(
    model=model,
    data=X_train,
    x=None,  # Will be set during explanation
    n_mc_samples=100,
    random_state=42,
)

# 2. Create explainer using imputer
explainer_gaussian = shapiq.TabularExplainer(
    model=model,
    data=X_train,
    index="SII",
    max_order=2,
    # attributes of the imputer
    n_features=n_features,
    n_mc_samples=100,
    imputer=gaussian_imputer,  # Pass custom imputer instance
    random_state=42,
)

# 3. Explain instance
x_explain = X_test[100]
gaussian_values = explainer_gaussian.explain(x_explain, budget=2**n_features, random_state=0)
print(gaussian_values)

ModuleNotFoundError: No module named 'shapiq_student'

### Explain Instance: 

In [ ]:
x_explain = X_test[100]
gaussian_values = explainer_gaussian.explain(x_explain, budget=2**n_features, random_state=0)
print(gaussian_values)

### Visualize Results: 

In [ ]:
shapiq.network_plot(
    interaction_values=gaussian_values,
    feature_names=X.columns,
    title="Shapley Interactions (Gaussian Imputation)",
)

## Gaussian Copula Imputation for Model Explanations
The following imputer is the `CopulaImputer` from `shapiq_student`. This imputer handles non-Gaussian features via ranktransformation.

### Initialize Copula Imputer

In [13]:
from shapiq_student import GaussianCopulaImputer

# 1. Create custom Copula imputer
copula_imputer = GaussianCopulaImputer(
    model=model,
    data=X_train,
    x=None,  # Will be set during explanation
    n_mc_samples=100,
    random_state=42,
)

# 2. Create explainer using imputer
explainer_copula = shapiq.TabularExplainer(
    model=model,
    data=X_train,
    index="SII",
    max_order=2,
    # attributes of the imputer
    n_features=n_features,
    n_mc_samples=100,
    imputer=copula_imputer,  # Pass custom imputer instance
    random_state=42,
)

# 3. Explain instance
x_explain = X_test[100]
copula_values = explainer_copula.explain(x_explain, budget=2**n_features, random_state=0)
print(copula_values)

ModuleNotFoundError: No module named 'shapiq_student'

### Explain Instance: 

In [ ]:
x_explain = X_test[100]
copula_values = explainer_copula.explain(x_explain, budget=2**n_features, random_state=0)
print(copula_values)

### Visualize Results: 

In [ ]:
shapiq.network_plot(
    interaction_values=copula_values,
    feature_names=X.columns,
    title="Shapley Interactions (Gaussian Copula Imputation)",
)

## Conclusion
- Both methods handle missing features via conditional imputation
- **Gaussian:** Fast but assumes normality
- **Gaussian Copula:** More general but computationally heavier
- Choice depends on data distribution!

## References

1. <a name="ref1"></a> Aas, Kjersti, Martin Jullum, and Anders Løland. *Explaining individual predictions when features are dependent: More accurate approximations to Shapley values.* Artificial Intelligence 298 (2021): 103502.

2. <a name="ref2"></a> Jullum, Martin, et al. *shapr: Explaining Machine Learning Models with Conditional Shapley Values in R and Python.* arXiv preprint arXiv:2504.01842 (2025).